# ESM C (Cambrian)

![ESM C (Cambrian)](https://proto-bio.github.io/proto-assets/images/tool/esmc/hero.png)

ESM C is an embedding-focused masked protein language model from Biohub. Unlike ESM3, ESM C is purpose-built for producing high-quality contextualized embeddings: there is no sample/score interface.

Three open-weights variants are exposed here, all MIT-licensed: `esmc_300m` (960-dim), `esmc_600m` (1152-dim), and `esmc_6b` (2560-dim). This notebook uses the 300M default; `esmc_6b` pulls roughly 25 GB of weights on first use.

The toolkit also exposes sparse autoencoders over ESM C's activations, covered in the second half of this notebook.


In [ ]:
from proto_tools.utils.notebook_docs import (
    display_overview, display_api_reference, display_doc_link, display_available_tools,
)
display_doc_link("esmc")
display_overview("esmc")

## Available tools

In [ ]:
display_available_tools("esmc")

### `run_esmc_embeddings`

Returns a `SequenceEmbedding` per input sequence containing a mean-pooled embedding, the attention mask, and (optionally) per-position amino-acid logits.

In [ ]:
import numpy as np

from proto_tools.tools.masked_models.esmc import (
    ESMCEmbeddingsConfig, ESMCEmbeddingsInput, run_esmc_embeddings,
)

# Two hemoglobin chains (alpha, beta) are close homologs; GFP is an unrelated control.
hba = "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHGKKVADALTNAVAHVDDMPNALSALSDLHAHKLRVDPVNFKLLSHCLLVTLAAHLPAEFTPAVHASLDKFLASVSTVLTSKYR"
hbb = "MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLSTPDAVMGNPKVKAHGKKVLGAFSDGLAHLDNLKGTFATLSELHCDKLHVDPENFRLLGNVLVCVLAHHFGKEFTPPVQAAYQKVVAGVANALAHKYH"
gfp = "MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTFSYGVQCFSRYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNYNSHNVYIMADKQKNGIKVNFKIRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGITHGMDELYK"

result = run_esmc_embeddings(ESMCEmbeddingsInput(sequences=[hba, hbb, gfp]), ESMCEmbeddingsConfig())
embs = [np.array(r.mean_embedding) for r in result.results]

def cosine(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

print(f"Embedding dim: {embs[0].shape[0]}")
print(f"cosine(HBA, HBB) = {cosine(embs[0], embs[1]):.3f}  (homologous globins)")
print(f"cosine(HBA, GFP) = {cosine(embs[0], embs[2]):.3f}  (unrelated)")

### `run_esmc_sae_features`

Sparse autoencoders decompose ESM C's internal activations into a large, sparsely-active feature space that is easier to interpret than raw embeddings. Each residue activates exactly `k` features out of a codebook of thousands. Every combination of layer, `k`, and `codebook_size` is a separately trained SAE rather than a runtime setting, and not all combinations were published.

A consequence: within one SAE a feature index always denotes the same learned concept, so activations are comparable across proteins. Indices are *not* comparable between different SAEs — including different layers of the same backbone.

This notebook uses the 300M backbone at its default layer. See the [SAE overview card](https://huggingface.co/biohub/ESMC-SAE-Overview) for the full set of variants.


In [ ]:
from proto_tools.tools.masked_models.esmc import (
    ESMCSAEFeaturesConfig, ESMCSAEFeaturesInput, run_esmc_sae_features,
)

# Human ubiquitin (P0CG48 residues 1-76) — small, well characterized.
ubiquitin = (
    "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG"
)
inputs = ESMCSAEFeaturesInput(sequences=[ubiquitin])
result = run_esmc_sae_features(inputs, ESMCSAEFeaturesConfig())

features = result.results[0].layers[0]
print(f"layer {features.layer}: {len(features.feature_indices)} residues, "
      f"{len(features.feature_indices[0])} active features each")

## Which features fire most consistently?

A feature active at many positions is tracking something global about the protein; one active at a single position is tracking a local motif.

In [ ]:
from collections import Counter

counts = Counter(idx for row in features.feature_indices for idx in row)
for feature_index, n in counts.most_common(5):
    print(f"feature {feature_index:>6}: active at {n:>3}/{len(features.feature_indices)} residues")

## Comparing layers

Attaching several SAEs in one call runs the backbone once. Only the requested layers are downloaded.

In [ ]:
multi = run_esmc_sae_features(
    inputs, ESMCSAEFeaturesConfig(layers=[5, 15, 23])
)
for layer in multi.results[0].layers:
    strongest = max(
        (m[0], i[0]) for i, m in zip(layer.feature_indices, layer.feature_magnitudes)
    )
    print(f"layer {layer.layer:>2}: strongest activation {strongest[0]:.2f} on feature {strongest[1]}")

## What do the features mean?

Biohub generated natural-language descriptions for one SAE — `ESMC-6B-sae-layer60-k64-codebook16384` — by analyzing how each feature activates across UniRef90. `describe_sae_features` fetches them from the public ESM Atlas API. Feature indices are specific to the SAE that produced them, so these apply only to features from that exact SAE; the 6B defaults resolve to it.

**Rank by normalized activation, not raw magnitude.** The strongest raw activations are dominated by features that fire on almost every protein and carry little information. Each record ships the statistics needed to correct for that: `(activation / uniref90_max_activation) * uniref90_idf` scales a feature to [0, 1] and upweights rare, distinctive ones. On ubiquitin, raw magnitude surfaces an "Unknown generic feature"; normalized, the top hit is a ubiquitin-like domain detector.

In [ ]:
from proto_tools.tools.masked_models.esmc import DESCRIBED_SAE_REPO, describe_sae_features

described_config = ESMCSAEFeaturesConfig(model_checkpoint="esmc_6b")
assert described_config.sae_repo == DESCRIBED_SAE_REPO

features_6b = run_esmc_sae_features(inputs, described_config).results[0].layers[0]

# Peak activation per feature across the sequence.
peak = {}
for indices, magnitudes in zip(features_6b.feature_indices, features_6b.feature_magnitudes):
    for feature_id, magnitude in zip(indices, magnitudes):
        peak[feature_id] = max(peak.get(feature_id, 0.0), magnitude)

# Over 1,200 features fire somewhere on this sequence. Describing the 50 strongest
# and re-ranking those gives the same answer as describing all of them.
candidates = sorted(peak, key=peak.get, reverse=True)[:50]
records = describe_sae_features(candidates)

def normalized(feature_id):
    """Scale to [0, 1] by the feature's UniRef90 maximum, then weight by its IDF."""
    record = records[feature_id]
    return (peak[feature_id] / record["uniref90_max_activation"]) * record["uniref90_idf"]

for feature_id in sorted(records, key=normalized, reverse=True)[:5]:
    record = records[feature_id]
    print(f"{normalized(feature_id):5.2f}  feature {feature_id:>5}  {record['label']}")
    print(f"       category={record['category']}")

### `run_esmc_gradient`

The gradient tool turns ESM C's naturalness score into a differentiable objective. It takes a relaxed `(L, 20)` distribution over amino acids and returns the gradient of the masked pseudo-log-likelihood with respect to those logits: each position is masked in turn and predicted from its bidirectional context, so the objective is the mean negative log-likelihood over all residues.

That makes it usable as the objective inside a sequence-design loop — an optimizer holds a relaxed distribution, asks for the gradient, and steps toward sequences ESM C finds more plausible. Pass `compute_gradient=False` for forward-only naturalness scoring, which is the cheaper call when ranking candidates rather than optimizing them.

In [ ]:
from proto_tools.tools.masked_models.esmc import (
    ESMCGradientConfig, ESMCGradientInput, run_esmc_gradient,
)
from proto_tools.utils import one_hot_protein_logits

display_api_reference("esmc", "input", "run_esmc_gradient")

# Seed a relaxed distribution from the first 60 residues of hemoglobin alpha.
# sharpness=2.0 gives a biased-but-not-saturated softmax target.
grad_logits = one_hot_protein_logits(hba[:60], sharpness=2.0)

grad_output = run_esmc_gradient(
    ESMCGradientInput(inputs=grad_logits, temperature=0.6),
    ESMCGradientConfig(),
)

# One result per candidate; this call passed one, so results holds one.
grad_candidate = grad_output.results[0]
grad_term = grad_candidate.terms[0]
print(f"Mean NLL:           {grad_term.term_value:.3f}")
print(f"Perplexity:         {grad_candidate.metrics['perplexity']:.3f}")
print(f"Gradient shape:     ({len(grad_term.gradient)}, {len(grad_term.gradient[0])})")
print(f"Gradient row 0[:5]: {[round(v, 5) for v in grad_term.gradient[0][:5]]}")


In [ ]:
# A batch scores several candidates in one call and fans them out across devices,
# returning one result per candidate in input order.
grad_batch = run_esmc_gradient(
    ESMCGradientInput(inputs=[one_hot_protein_logits(seq[:60], sharpness=2.0) for seq in (hba, hbb, gfp)]),
    ESMCGradientConfig(compute_gradient=False),  # forward-only: score, don't optimize
)
for grad_name, grad_result in zip(("hba", "hbb", "gfp"), grad_batch.results):
    grad_term = grad_result.terms[0]
    print(f"{grad_name}: mean NLL {grad_term.term_value:.3f}  (gradient={grad_term.gradient})")


## Export

In [ ]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)
result.export(name="esmc_sae_features", export_path=output_dir, file_format="csv")
print(sorted(p.name for p in output_dir.glob("esmc_sae_features*")))

## API reference

In [ ]:
display_api_reference("esmc", "input", "run_esmc_embeddings")
display_api_reference("esmc", "config", "run_esmc_embeddings")
display_api_reference("esmc", "output", "run_esmc_embeddings")
display_api_reference("esmc", "input", "run_esmc_sae_features")
display_api_reference("esmc", "config", "run_esmc_sae_features")
display_api_reference("esmc", "output", "run_esmc_sae_features")
display_api_reference("esmc", "input", "run_esmc_gradient")
display_api_reference("esmc", "config", "run_esmc_gradient")
display_api_reference("esmc", "output", "run_esmc_gradient")
